[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagedSaeed/instructions-tuning/blob/main/Notebooks/Experiments/Baselines/emotion_detection_tuning.ipynb)

# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# !echo $GT_TOKEN # makesure the token is loaded

In [4]:
if 'jrcai_corekit' not in os.listdir('.'):
    !git clone https://${GT_TOKEN}@github.com/MagedSaeed/jrcai_corekit.git
else: # else, pull latest changes
    !cd jrcai_corekit && git pull && cd ..

Already up to date.


In [5]:
!pip install -r jrcai_corekit/requirements.txt

add jrcai_corekit to path

In [6]:
import sys
sys.path.append('/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit')

check everything is working

In [7]:
from llm.text_generator import TextGenerator

/home/majed_alshaibani/Projects/InstructionsTuning/venv/lib/python3.10/site-packages/transformers/deepspeed.py:24: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(


# Building the prompts dataset

In [ ]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14776,
  'tags': [],
  'name': 'MaisPrompt',
  'task': {'name': 'question answering (yes/no)'},
  'status': 'SUBMITTED',
  'template': "You're tasked to answer a cultural question about Arabs, the question is a statement written in Arabic, and your goal is to affirm or deny.\r\n\r\nYour only choices for the answer are: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %} or {% endif %}{% endfor %}.\r\n\r\nOutput 0 if the statement is wrong, and 1 if it's correct.\r\n\r\nStatement: {{question}}\r\nYour answer:\r\n|||\r\n{{answer_choices[answer]}}",
  'dataset_name': 'arbml/ACVA',
  'dataset_subset': 'default',
  'answer_choices': ['خطأ', 'صح'],
  'text_direction': 'ltr'},
 {'id': 14775,
  'tags': [],
  'name': 'MathsPrompt',
  'task': {'name': 'math solving'},
  'status': 'SUBMITTED',
  'template': 'Write the mathematical equation that represents the question in English, without additional explanations, or formatting. The final answer is not required, only a solv

In [14]:
len(prompts)

245

filter prompts:
- get only the approved ones
- get only the ones on the sarcasim detection datasets (emotone_ar,sem_eval_2018_task_1)

In [15]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED', prompts))
len(filtered_prompts)

117

## Finetuning emotone_ar dataset

### Get the dataset prompts

In [16]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: 'emotone_ar' in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(filtered_prompts)

117

### Download the dataset

In [17]:
import datasets

In [18]:
emotone_ar_experimental = datasets.load_dataset('MagedSaeed/emotone_ar_experimental')
emotone_ar_experimental

DatasetDict({
    train: Dataset({
        features: ['tweet', 'label'],
        num_rows: 8052
    })
    test: Dataset({
        features: ['tweet', 'label'],
        num_rows: 2013
    })
})

### Merge the prompts

In [19]:
from jinja2 import Environment, StrictUndefined

In [46]:
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    prefix = prefix.strip()
    suffix = suffix.strip()
    if prefix.endswith('.'):
        prefix = prefix[:-1]
    return f'{prefix} {suffix}'

In [47]:
def apply_template(prompt_template, sample):
    template = prompt_template['template']
    template = preprocess_template(template)
    sample['answer_choices'] = prompt_template['answer_choices']
    env = Environment(undefined=StrictUndefined)
    template = env.from_string(template)
    rendered_template = template.render(**sample)
    return rendered_template

see how the template is applied on different examples

### Perform generation on one example prompt, for experimentation

In [48]:
example_prompt_template = dataset_prompts[4]
print(apply_template(example_prompt_template, emotone_ar_experimental['train'][2]))

Review this tweet المشكله ليست فيمن يخذلك ، يخونك ، يوجعك ، يسحقك ، المشكله هي انك تتمكن من تصديق شخص نال منك مره لتمنحه فرصه النيل منك اخري ! carefully and then identify the emotion being expressed. Choose from the following options:none or anger or joy or sadness or love or sympathy or surprise or fear sadness


In [54]:
rendered_train_prompts_dataset = list(
    map(
        lambda sample: apply_template(example_prompt_template, sample),
        tqdm(emotone_ar_experimental['train'].select(range(min(len(emotone_ar_experimental['train']), 10_000)))),
        )
    )
len(rendered_train_prompts_dataset)

  0%|          | 0/8052 [00:00<?, ?it/s]

8052

## Finetune the LLM

In [55]:
GLOBAL_SEED = 42

In [56]:
import random
random.seed(GLOBAL_SEED)

In [68]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, JAISInitializer,LoRAConfigRepository
from sklearn.model_selection import train_test_split

In [58]:
import torch
MODEL_PATH = '/hdd/shared_models/jais-13b'
TOKENIZER_PATH = MODEL_PATH

In [59]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=JAISInitializer(),
)
llm_loader

In [60]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /hdd/shared_models/jais-13b/config.json
loading configuration file /hdd/shared_models/jais-13b/config.json
Model config JAISConfig {
  "_name_or_path": "/hdd/shared_models/jais-13b",
  "activation_function": "swiglu",
  "architectures": [
    "JAISLMHeadModel"
  ],
  "attn_pdrop": 0.0,
  "auto_map": {
    "AutoConfig": "configuration_jais.JAISConfig",
    "AutoModel": "modeling_jais.JAISModel",
    "AutoModelForCausalLM": "modeling_jais.JAISLMHeadModel",
    "AutoModelForQuestionAnswering": "modeling_jais.JAISForQuestionAnswering",
    "AutoModelForSequenceClassification": "modeling_jais.JAISForSequenceClassification",
    "AutoModelForTokenClassification": "modeling_jais.JAISForTokenClassification"
  },
  "bos_token_id": 0,
  "embd_pdrop": 0.0,
  "embeddings_scale": 14.6,
  "eos_token_id": 0,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "jais",
  "n_embd": 5120,
  "n_head": 40,
  "n_inner": 13653,
  "n_layer": 40,
  "n_position

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

All model checkpoint weights were used when initializing JAISLMHeadModel.

All the weights of JAISLMHeadModel were initialized from the model checkpoint at /hdd/shared_models/jais-13b.
If your task is similar to the task the model of the checkpoint was trained on, you can already use JAISLMHeadModel for predictions without further training.
loading configuration file /hdd/shared_models/jais-13b/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 0,
  "eos_token_id": 0,
  "pad_token_id": 0
}

loading file tokenizer.json
loading file tokenizer.model
loading file added_tokens.json
loading file special_tokens_map.json
loading file tokenizer_config.json
loading configuration file /hdd/shared_models/jais-13b/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 0,
  "eos_token_id": 0,
  "pad_token_id": 0
}



In [75]:
train_samples,eval_samples = train_test_split(rendered_train_prompts_dataset, test_size=0.1, random_state=GLOBAL_SEED)
def generate_tuple(sample):
    sample_words = sample.split(' ')
    prefix,suffix = ' '.join(sample_words[:-1]),f' {sample_words[-1]}'
    return prefix,suffix
train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(7246,
 806,
 [('Review this tweet don: كلنا معكم يا قلوبنا  رساله لجندي عراقي carefully and then identify the emotion being expressed. Choose from the following options:none or anger or joy or sadness or love or sympathy or surprise or fear',
   ' sympathy'),
  ('Review this tweet اسود وزن فوق 105 كيلو عاملين مذبحه في الاوليمبياد دلوئتي — watching the Olympic Games carefully and then identify the emotion being expressed. Choose from the following options:none or anger or joy or sadness or love or sympathy or surprise or fear',
   ' none'),
  ('Review this tweet ان شاء الله نصبح علي خير :) carefully and then identify the emotion being expressed. Choose from the following options:none or anger or joy or sadness or love or sympathy or surprise or fear',
   ' joy'),
  ('Review this tweet فريق الهاند بول اثبت بعد الاوليمبياد انه فريق ساذج و ينقصه الاعداد النفسي و العقليه اللي تخليه يلعب علي بطوله  يتبع carefully and then identify the emotion being expressed. Choose from the following optio

In [76]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.jais_v1(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=16,
    eval_batch_size=16,
    output_dir='./tuned_models/jais-13b/'
)

/home/majed_alshaibani/Projects/InstructionsTuning/venv/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.
Using auto half precision backend
***** Running training *****
  Num examples = 7,246
  Num Epochs = 10
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total 

Epoch,Training Loss,Validation Loss
1,No log,0.337350
2,0.467800,0.362700
3,0.225300,0.372148
4,0.110500,0.660484
5,0.037400,0.808043
6,0.007200,0.812085



***** Running Evaluation *****
  Num examples = 806
  Batch size = 16
loading configuration file /hdd/shared_models/jais-13b/config.json
Model config JAISConfig {
  "_name_or_path": "inception-mbzuai/jais-13b",
  "activation_function": "swiglu",
  "architectures": [
    "JAISLMHeadModel"
  ],
  "attn_pdrop": 0.0,
  "auto_map": {
    "AutoConfig": "configuration_jais.JAISConfig",
    "AutoModel": "modeling_jais.JAISModel",
    "AutoModelForCausalLM": "modeling_jais.JAISLMHeadModel",
    "AutoModelForQuestionAnswering": "modeling_jais.JAISForQuestionAnswering",
    "AutoModelForSequenceClassification": "modeling_jais.JAISForSequenceClassification",
    "AutoModelForTokenClassification": "modeling_jais.JAISForTokenClassification"
  },
  "bos_token_id": 0,
  "embd_pdrop": 0.0,
  "embeddings_scale": 14.6,
  "eos_token_id": 0,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "jais",
  "n_embd": 5120,
  "n_head": 40,
  "n_inner": 13653,
  "n_layer": 40,
  "n_positi

{'eval_loss': 0.33734965324401855, 'eval_runtime': 21.9352, 'eval_samples_per_second': 36.745, 'eval_steps_per_second': 2.325, 'epoch': 1.0}



***** Running Evaluation *****
  Num examples = 806
  Batch size = 16


...

{'eval_loss': 0.8080429434776306, 'eval_runtime': 21.9271, 'eval_samples_per_second': 36.758, 'eval_steps_per_second': 2.326, 'epoch': 5.0}



***** Running Evaluation *****
  Num examples = 806
  Batch size = 16


{'eval_loss': 0.8120847344398499, 'eval_runtime': 21.936, 'eval_samples_per_second': 36.743, 'eval_steps_per_second': 2.325, 'epoch': 6.0}




Training completed. Do not forget to share your model on huggingface.co/models =)




0.33734965324401855